# Stock Price Prediction

Predict the **next trading day's closing price** from historical OHLCV data.

A literal `Price` column is not required: `Close` is the price series used as the forecasting target.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
csvs=sorted(Path("data").glob("*.csv"))
if not csvs: raise FileNotFoundError("Put an OHLCV CSV in data/")
df=pd.read_csv(csvs[0])
df.columns=[c.strip().lower().replace(" ","_") for c in df.columns]
df.head()

,date,symbol,series,prev_close,open,high,low,last,close,vwap,volume,turnover,trades,deliverable_volume,%deliverble
0,2000-01-03,RELIANCE,EQ,233.05,237.50,251.70,237.50,251.70,251.70,249.37,4456424,1.111319e+14,NaN,NaN,NaN
1,2000-01-04,RELIANCE,EQ,251.70,258.40,271.85,251.30,271.85,271.85,263.52,9487878,2.500222e+14,NaN,NaN,NaN
2,2000-01-05,RELIANCE,EQ,271.85,256.65,287.90,256.65,286.75,282.50,274.79,26833684,7.373697e+14,NaN,NaN,NaN
3,2000-01-06,RELIANCE,EQ,282.50,289.00,300.70,289.00,293.50,294.35,295.45,15682286,4.633254e+14,NaN,NaN,NaN
4,2000-01-07,RELIANCE,EQ,294.35,295.00,317.90,293.00,314.50,314.55,308.91,19870977,6.138388e+14,NaN,NaN,NaN


In [2]:
aliases={
 "date":["date","datetime","timestamp"],"open":["open","open_price"],
 "high":["high","high_price"],"low":["low","low_price"],
 "close":["close","adj_close","adjusted_close","last","price"],
 "volume":["volume","vol","total_volume"]
}
resolved={}
for k,names in aliases.items():
    for n in names:
        if n in df.columns: resolved[k]=n; break
if "date" not in resolved or "close" not in resolved:
    raise ValueError(f"Need Date + Close/Price. Found: {list(df.columns)}")
for k,c in resolved.items():
    if k=="date": df[c]=pd.to_datetime(df[c],errors="coerce")
    else: df[c]=pd.to_numeric(df[c].astype(str).str.replace(",","",regex=False),errors="coerce")
df=df.dropna(subset=[resolved["date"],resolved["close"]]).sort_values(resolved["date"]).drop_duplicates(resolved["date"]).reset_index(drop=True)

In [3]:
c=resolved["close"]
df["return_1"]=df[c].pct_change()
df["return_5"]=df[c].pct_change(5)
for n in [1,2,3,5]: df[f"lag_{n}"]=df[c].shift(n)
for n in [5,10,20]: df[f"sma{n}_gap"]=df[c]/df[c].rolling(n).mean()-1
df["volatility10"]=df["return_1"].rolling(10).std()
df["high_low_pct"]=(df[resolved["high"]]-df[resolved["low"]])/df[c] if "high" in resolved and "low" in resolved else 0
df["open_close_pct"]=(df[c]-df[resolved["open"]])/df[resolved["open"]] if "open" in resolved else 0
df["volume_change"]=df[resolved["volume"]].pct_change() if "volume" in resolved else 0
df["target_next_close"]=df[c].shift(-1)
features=["lag_1","lag_2","lag_3","lag_5","return_1","return_5","sma5_gap","sma10_gap","sma20_gap","volatility10","high_low_pct","open_close_pct","volume_change"]
m=df[features+["target_next_close"]].replace([np.inf,-np.inf],np.nan).dropna()

In [4]:
split=int(len(m)*.8)
model=Pipeline([("scale",StandardScaler()),("ridge",Ridge(alpha=1.0))])
model.fit(m[features].iloc[:split],m.target_next_close.iloc[:split])
p=model.predict(m[features].iloc[split:])
print("MAE:",mean_absolute_error(m.target_next_close.iloc[split:],p))
print("RMSE:",np.sqrt(mean_squared_error(m.target_next_close.iloc[split:],p)))
print("R2:",r2_score(m.target_next_close.iloc[split:],p))

MAE: 21.42581635398225
RMSE: 39.52462041709221
R2: 0.989605110491199


## Final forecast

After evaluation, retrain on all usable samples and predict the next trading day's close from the latest available row.

In [5]:
model.fit(m[features],m.target_next_close)
latest=df.iloc[-1]
# The latest row has no target, so rebuild its features explicitly.
def latest_features(df):
    c=resolved["close"]; i=len(df)-1
    vals=df[c].to_numpy()
    ret1=vals[i]/vals[i-1]-1; ret5=vals[i]/vals[i-5]-1
    rs=[vals[j]/vals[j-1]-1 for j in range(i-9,i+1)]
    return np.array([
      vals[i-1],vals[i-2],vals[i-3],vals[i-5],ret1,ret5,
      vals[i]/np.mean(vals[i-4:i+1])-1,
      vals[i]/np.mean(vals[i-9:i+1])-1,
      vals[i]/np.mean(vals[i-19:i+1])-1,
      np.std(rs),
      ((df[resolved["high"]].iloc[i]-df[resolved["low"]].iloc[i])/vals[i]) if "high" in resolved and "low" in resolved else 0,
      ((vals[i]-df[resolved["open"]].iloc[i])/df[resolved["open"]].iloc[i]) if "open" in resolved else 0,
      (df[resolved["volume"]].iloc[i]/df[resolved["volume"]].iloc[i-1]-1) if "volume" in resolved else 0
    ]).reshape(1,-1)
pred=float(model.predict(latest_features(df))[0])
print("Latest close:",float(latest[c]))
print("Predicted next close:",pred)
print("Direction:","UP" if pred>=float(latest[c]) else "DOWN")

Latest close: 1994.5
Predicted next close: 1996.2604471540803
Direction: UP
